# 06 - Inference & Demo

Interactive inference with the trained ModernBERT-RGAT model:

1. Load trained model
2. Single text prediction
3. Batch prediction on example reviews
4. Visual aspect highlighting
5. Custom text input

---

## 1. Setup

In [1]:
import subprocess, sys, os

def install_if_missing(package, pip_name=None):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or package])

install_if_missing('transformers')
install_if_missing('spacy')

import spacy
try:
    spacy.load('en_core_web_sm')
except OSError:
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])

print('Dependencies ready.')

/home/thota23/miniconda3/envs/bash_ai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dependencies ready.


In [2]:
PROJECT_ROOT = os.path.expanduser('~/SOTA-ModernBERT-RGAT-Joint-Aspect-Sentiment-Extraction-for-Food-Tech-Reviews')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import torch
from IPython.display import HTML, display
from src.inference import AspectSentimentPredictor, load_predictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Imports successful.')

Device: cuda
Imports successful.


## 2. Load Trained Model

In [3]:
# Load the best available checkpoint
# Try 2014 first (most common), then 2015, then 2016
predictor = None
for year in ['2014', '2015', '2016']:
    ckpt = f'checkpoints/best_model_{year}.pt'
    if os.path.exists(ckpt):
        print(f'Loading model from: {ckpt}')
        predictor = load_predictor(year=year, device=device)
        print(f'Model loaded successfully on {device}!')
        break

if predictor is None:
    print('ERROR: No checkpoints found. Train a model first using 04_training.ipynb')

Loading model from: checkpoints/best_model_2014.pt


Loading weights: 100%|██████████| 134/134 [00:00<00:00, 492.12it/s, Materializing param=layers.21.mlp_norm.weight]     
ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/thota23/SOTA-ModernBERT-RGAT-Joint-Aspect-Sentiment-Extraction-for-Food-Tech-Reviews/src/inference.py:141: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_o

Loaded checkpoint: /home/thota23/SOTA-ModernBERT-RGAT-Joint-Aspect-Sentiment-Extraction-for-Food-Tech-Reviews/checkpoints/best_model_2014.pt
  Epoch: 6
Model loaded successfully on cuda!


## 3. Single Text Prediction

In [4]:
text = "The pasta was absolutely delicious but the service was terribly slow."

predictions = predictor.predict(text)
print(predictor.format_predictions(text, predictions))

Text: "The pasta was absolutely delicious but the service was terribly slow."
  Found 1 aspect(s):
    😊 "pasta" → positive (conf: 0.84, chars 3:9)


In [5]:
# Visual HTML highlighting
html = predictor.get_highlighted_html(text, predictions)

legend = '''
<div style="margin-top: 12px; font-size: 13px;">
  <b>Legend:</b>
  <span style="background: #27ae60; color: white; padding: 2px 8px; border-radius: 4px;">Positive</span>
  <span style="background: #e74c3c; color: white; padding: 2px 8px; border-radius: 4px;">Negative</span>
  <span style="background: #f39c12; color: white; padding: 2px 8px; border-radius: 4px;">Neutral</span>
  <span style="background: #8e44ad; color: white; padding: 2px 8px; border-radius: 4px;">Conflict</span>
</div>
'''

display(HTML(f'<div style="font-size: 18px; line-height: 2; padding: 16px; background: #f8f9fa; border-radius: 8px;">{html}</div>{legend}'))

## 4. Batch Prediction — Restaurant Reviews

In [6]:
example_reviews = [
    "The spicy ramen was incredibly flavorful and the broth was rich.",
    "Terrible pizza with a soggy crust, but the drinks were excellent.",
    "Average food, nothing special about the ambiance either.",
    "The sushi here is the best I have ever had, fresh and perfectly seasoned.",
    "Long wait times and rude staff ruined an otherwise decent meal.",
    "Loved the cheesecake but the coffee was lukewarm and bitter.",
    "Great location with a beautiful patio, although prices are a bit high.",
    "The butter chicken was creamy and aromatic, paired perfectly with garlic naan.",
]

print('=' * 70)
print(' ModernBERT-RGAT: Aspect Sentiment Predictions')
print('=' * 70)

for i, review in enumerate(example_reviews, 1):
    preds = predictor.predict(review)
    print(f'\n[{i}] {predictor.format_predictions(review, preds)}')
    print('-' * 70)

 ModernBERT-RGAT: Aspect Sentiment Predictions

[1] Text: "The spicy ramen was incredibly flavorful and the broth was rich."
  Found 2 aspect(s):
    😊 "ram" → positive (conf: 0.50, chars 9:13)
    😊 "broth" → positive (conf: 0.61, chars 48:54)
----------------------------------------------------------------------

[2] Text: "Terrible pizza with a soggy crust, but the drinks were excellent."
  Found 2 aspect(s):
    😞 "pizza" → negative (conf: 0.91, chars 8:14)
    😊 "drinks" → positive (conf: 0.89, chars 42:49)
----------------------------------------------------------------------

[3] Text: "Average food, nothing special about the ambiance either."
  Found 2 aspect(s):
    😐 "food" → neutral (conf: 0.80, chars 7:12)
    😞 "am" → negative (conf: 0.36, chars 39:42)
----------------------------------------------------------------------

[4] Text: "The sushi here is the best I have ever had, fresh and perfectly seasoned."
  Found 1 aspect(s):
    😊 "sushi" → positive (conf: 0.87, chars 3

In [7]:
# Visual HTML for all reviews
html_blocks = []
for review in example_reviews:
    preds = predictor.predict(review)
    highlighted = predictor.get_highlighted_html(review, preds)
    html_blocks.append(
        f'<div style="padding: 10px 16px; margin: 8px 0; '
        f'background: #f8f9fa; border-radius: 8px; '
        f'font-size: 16px; line-height: 2; '
        f'border-left: 4px solid #3498db;">{highlighted}</div>'
    )

all_html = '<h3 style="color: #2c3e50;">🍽️ Restaurant Review Analysis</h3>' + '\n'.join(html_blocks) + legend
display(HTML(all_html))

## 5. Prediction Details

In [8]:
import pandas as pd

# Collect all predictions into a table
rows = []
for review in example_reviews:
    preds = predictor.predict(review)
    if preds:
        for p in preds:
            rows.append({
                'Review': review[:50] + '...' if len(review) > 50 else review,
                'Aspect': p.aspect,
                'Sentiment': p.sentiment,
                'Confidence': f'{p.confidence:.3f}',
                'Position': f'{p.start}:{p.end}',
            })
    else:
        rows.append({
            'Review': review[:50] + '...' if len(review) > 50 else review,
            'Aspect': '—',
            'Sentiment': '—',
            'Confidence': '—',
            'Position': '—',
        })

pred_df = pd.DataFrame(rows)
display(pred_df.style.set_caption('Prediction Details'))

,Review,Aspect,Sentiment,Confidence,Position
0,The spicy ramen was incredibly flavorful and the b...,ram,positive,0.502,9:13
1,The spicy ramen was incredibly flavorful and the b...,broth,positive,0.608,48:54
2,"Terrible pizza with a soggy crust, but the drinks ...",pizza,negative,0.909,8:14
3,"Terrible pizza with a soggy crust, but the drinks ...",drinks,positive,0.890,42:49
4,"Average food, nothing special about the ambiance e...",food,neutral,0.798,7:12
5,"Average food, nothing special about the ambiance e...",am,negative,0.359,39:42
6,"The sushi here is the best I have ever had, fresh ...",sushi,positive,0.874,3:9
7,Long wait times and rude staff ruined an otherwise...,staff,negative,0.977,24:30
8,Long wait times and rude staff ruined an otherwise...,meal,neutral,0.560,57:62
9,Loved the cheesecake but the coffee was lukewarm a...,chees,positive,0.453,9:15


## 6. Try Your Own Text

In [9]:
# Type your own review here!
custom_text = "The margherita pizza had a perfectly crispy crust but the toppings were bland."

preds = predictor.predict(custom_text)
print(predictor.format_predictions(custom_text, preds))
print()

html = predictor.get_highlighted_html(custom_text, preds)
display(HTML(f'<div style="font-size: 18px; line-height: 2; padding: 16px; background: #f8f9fa; border-radius: 8px;">{html}</div>{legend}'))

Text: "The margherita pizza had a perfectly crispy crust but the toppings were bland."
  No aspects detected.



## 7. Export Predictions

In [10]:
import json

# Export predictions as JSON
export = []
for review in example_reviews:
    preds = predictor.predict(review)
    export.append({
        'text': review,
        'predictions': [p.to_dict() for p in preds],
    })

os.makedirs('outputs/results', exist_ok=True)
with open('outputs/results/inference_examples.json', 'w') as f:
    json.dump(export, f, indent=2)

print('Predictions exported to: outputs/results/inference_examples.json')
print(f'Total reviews: {len(export)}')
print(f'Total aspects found: {sum(len(e["predictions"]) for e in export)}')

Predictions exported to: outputs/results/inference_examples.json
Total reviews: 8
Total aspects found: 14


---

## Phase 6 Summary

| Component | Status |
|-----------|--------|
| Inference pipeline (`src/inference.py`) | Done |
| Per-aspect sentiment (individual masks) | Done |
| HTML visualization with sentiment colors | Done |
| Batch prediction | Done |
| JSON export | Done |
| Gradio web demo (`app.py`) | Done |

**Next step:** Phase 7 - Documentation & Packaging

> **Note:** Prediction quality depends on training quality. Retrain with full fine-tuning when GPU is ready for better results.